## Introduction to Neuromorphic Control - Capstone Project

**Students:**
- Samuel Galla
- Tim Huber

### Python Setup

In [1]:
import mujoco
import mujoco.viewer

import numpy as np
import mediapy as media
import matplotlib.pyplot as plt

### MuJoCo Setup
The following cell contains the code for running the mujoco visualizer with our current scene.

In [2]:
# Load xml from file
with open("world.xml", "r") as f:
    xml = f.read()

# Add a sphere body to the world
spec = mujoco.MjSpec.from_file("world.xml")
model = spec.compile()
data = mujoco.MjData(model)

# mujoco.viewer.launch(model, data)

In [3]:
# Test cell for adding a sphere to the world
body = spec.worldbody.add_body(
    name="ball_body",
    pos=[0, 0, 2.0],
    quat=[0, 0, 0, 0],
    mass=0.00167
)

# Add a sphere geom to the body
geom = body.add_geom(
    type=mujoco.mjtGeom.mjGEOM_SPHERE,
    name="ball_geom",
    size=[0.02, 0, 0],
    rgba=[1, 0.5, 0, 1],
    # solref=[0.02, 0.5]
)

# Add a free joint to the body
joint = body.add_joint(
    name="ball",
    type=mujoco.mjtJoint.mjJNT_FREE,
    damping=0,
)

# Setup the Pairs
neue_verbindungen = [
    ("table_collider_v", "ball_geom"),     # (Geom 1, Geom 2)
    ("table_collider_v", "ball_geom"),
    ("ground", "ball_geom")
]

for i, (g1, g2) in enumerate(neue_verbindungen):
    pair = spec.add_pair()
    pair.name = f"custom_pair_{i}" 
    pair.geomname1 = g1             
    pair.geomname2 = g2 

model, data = spec.recompile(model, data)

In [4]:
mujoco.viewer.launch(model, data)

In [ ]:
robot_joints = ["base_x", "base_y", "rotator1", "rotator2", "arm1", "arm2", "paddle_rotator", "paddle"]

joint_ids = [data.joint(j).id for j in robot_joints]
motor_ids = [data.actuator(m).id for m in robot_joints]

# Get bias forces for the actuators
bias_forces = data.qfrc_bias[motor_ids]

# Get body weights for the robot arm
body_weights = [data.body(j) for j in ["base"] + robot_joints[2:]]

print("Joint IDs:", joint_ids)
print("Motor IDs:", motor_ids)
print()
print("Joint positions:", data.qpos[joint_ids])
print("Motor controls:", data.ctrl[motor_ids])
print()
print("Bias forces:", bias_forces)
print()
print("Body weights:", model.body_mass[[b.id for b in body_weights]])


Joint IDs: [0, 1, 2, 3, 4, 5, 6, 7]
Motor IDs: [0, 1, 2, 3, 4, 5, 6, 7]

Joint positions: [0. 0. 0. 0. 0. 0. 0. 0.]
Motor controls: [0. 0. 0. 0. 0. 0. 0. 0.]

Bias forces: [0. 0. 0. 0. 0. 0. 0. 0.]

Body weights: [12.5   0.25  0.25  0.55  0.45  0.1   0.15]


### Plan
1. **Basic Limb Control**
    - (Fast) Controllers for all 8 Joints
    - Basic inverse kinematics for positioning and angeling
    - Maybe expand to more advanced positioning techniques?

2. **Bouncing the Ball**
    - Ball spawner for random balls flying towards the robot (maybe with a bounce?)
    - Move the paddle to the postion of an incoming ball (so the ball bounces off the paddle)
    - Align paddle surface othogonal to the balls' velocity vector?

3. **Target Practice**
    - Bounce balls towards specific targets

4. **Hitting the Ball**
    - Move the paddle so it applies a specific force to the ball (giving the ball a specific velocity vector)
    - Make the ball hit targets that are not necessarely hittable by bouncing it off the paddle